# Part 1: Build knowledge bases with agentic retrieval

Part 1 at Caldova begins with a procurement analyst investigating supplier risk. You need one knowledge system that can retrieve purchasing policy, supplier evidence, and approved sourcing commitments without blurring the role of each source.

**🎯 Mission**
- Upload Caldova's purchasing policy directly to a **File Knowledge Source** and query its controls
- Connect the **supplier evidence** and **sourcing documents** indexes as **Indexed Knowledge Sources**
- Build a multi-source **Knowledge Base** that routes procurement questions to the right evidence

## The building blocks

In Foundry IQ (Azure AI Search), a **knowledge base** is a top-level resource that connects a chat completion model to one or more knowledge sources. It defines which data sources to query, which model to use for reasoning, and how query execution should be optimized.

In this notebook, you'll explore two types of knowledge sources:

1. **File Knowledge Source**: Upload raw files directly. The service handles chunking, embedding, and indexing automatically.
2. **Indexed Knowledge Sources**: Connect to pre-built search indexes for production workloads with full control over schema and processing.

## Pre-configured environment

This lab uses the following Azure resources, already set up for you:

* Foundry IQ (Azure AI Search)
  * `supplier-evidence` index: 31 chunks from invoices, shipment evidence, production yield records, and quality records
  * `sourcing-documents` index: 16 chunks from RFP responses, agreements, amendments, and purchase orders
  * Semantic ranking enabled with pre-computed embeddings
* Azure OpenAI from Microsoft Foundry Models
  * `gpt-5.4`: Chat completion for reasoning and synthesis
  * `text-embedding-3-large`: Embedding model for vector search
* Microsoft Fabric

## Step 1: Set up environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has everything you need: Foundry IQ (Azure AI Search) endpoints, Azure OpenAI credentials, and model configuration.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"

SUPPLIER_EVIDENCE_INDEX = "supplier-evidence"
SOURCING_DOCUMENTS_INDEX = "sourcing-documents"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

---

## A: File Knowledge Source

A **File Knowledge Source** lets you upload raw files directly. The service handles chunking, embedding, and indexing automatically. This is the fastest way to get documents into a knowledge base.

## Step 2: Create a File Knowledge Source

Configure a file knowledge source with an embedding model for automatic ingestion. The `ingestion_parameters` specify:
- **Embedding model**: Generates vector embeddings for the uploaded content
- **Content extraction mode**: How to extract text from documents

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "file-docs-knowledge-source"

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

file_ks = FileKnowledgeSource(
    name=FILE_KNOWLEDGE_SOURCE_NAME,
    description="Caldova purchasing policy covering approval thresholds, sourcing routes, supplier governance, and escalation responsibilities",
    file_parameters=FileKnowledgeSourceParameters(
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                    api_key=AZURE_OPENAI_KEY,
                )
            ),
            content_extraction_mode="minimal",
            )
    ),
)

index_client.create_or_update_knowledge_source(file_ks)
print(f"File knowledge source '{FILE_KNOWLEDGE_SOURCE_NAME}' created or updated successfully.")

## Step 3: Upload files to the knowledge source

Upload a document directly to the file knowledge source. The service will automatically extract text, chunk it into searchable segments, and generate vector embeddings.

⏱️ The sample file takes around 40 seconds to ingest.

In [ ]:
from pathlib import Path

data_dir = Path("../..") / "data"
files_to_upload = [
    data_dir / "caldova" / "pdfs" / "CAL-POL-PUR-001.pdf"
]

uploaded_files = []
for file_path in files_to_upload:
    with open(file_path, "rb") as f:
        file_content = f.read()
    uploaded = index_client.upload_knowledge_source_file(
        FILE_KNOWLEDGE_SOURCE_NAME, file_content, filename=file_path.name
    )
    uploaded_files.append(uploaded)
    print(f"Uploaded: {file_path.name} (file_id: {uploaded.file_id}, {uploaded.file_size_bytes} bytes)")

print(f"\nTotal files uploaded: {len(uploaded_files)}")

## Step 4: Create a knowledge base for the file source

A knowledge base is the orchestration layer that combines:

1. Your data sources (the file knowledge source)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format responses

This knowledge base uses `ANSWER_SYNTHESIS` output mode, so the attached LLM will generate a complete natural language answer from the retrieved chunks.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
)

FILE_KNOWLEDGE_BASE_NAME = "caldova-policy-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=FILE_KNOWLEDGE_BASE_NAME,
    description="Caldova purchasing policy knowledge base",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=FILE_KNOWLEDGE_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{FILE_KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 5: Query the file knowledge base

Ask a question about the uploaded document. The knowledge base will search across the file content and synthesize a natural language answer with citations.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

file_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=FILE_KNOWLEDGE_BASE_NAME, credential=search_credential
)

file_ks_params = FileKnowledgeSourceParams(
    knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = "What approval thresholds, sourcing routes, and escalation responsibilities does Caldova's purchasing policy define?"

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(text=question)
            ],
        )
    ],
    knowledge_source_params=[file_ks_params],
    include_activity=True,
)

result = file_kb_client.retrieve(retrieval_request=req)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

The activity log shows the sequence of steps taken by the knowledge base. Each activity contains:

* `type`: The type of activity. For this log, you will see the first step of "modelQueryPlanning", multiple "file" steps (one for each query sent to the file knowledge source), "modelAnswerSynthesis" for the answer synthesis step, and "agenticReasoning" for the semantic re-ranking step.
* `elapsedMS`: The duration of the activity.
* `id`: The activity ID, which is helpful to connect each reference to the activity that produced it. You'll review references next.

The activity entries also contain other metadata specific to their type, so that you can see how many tokens were used by any model-powered steps and see the queries sent to sources for retrieval steps.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

The next cell prints the raw `references` array returned by the knowledge base.

The references show which sources were retrieved, and the source metadata for each reference. Every result contains:

* `type`: Always "file", since this knowledge base only contains a file source.
* `id`: A numeric identifier for the reference. This is what appears in the answer as citation markers like `[ref:1]`.
* `activitySource`: The id of the retrieval activity in the activity log that produced this reference. You can use this to trace which search step returned this document.
* `sourceData`: Fields from the retrieved chunk in the auto-created search index - always `uid`, `snippet`, and `metadata_storage_path` for file source indexes.
* `rerankerScore`: A score between 0-4 from the re-ranking model, where 4 is the best.
* `docName`: The auto-generated filename for the uploaded file.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

---

## B: Indexed knowledge sources

**Indexed Knowledge Sources** connect to pre-built search indexes, giving you full control over schema, chunking, and embedding. This is ideal for production workloads with large datasets.

For this section, two restored search indexes already contain document chunks. `supplier-evidence` holds operational evidence such as invoices, shipment records, yield records, and quality findings. `sourcing-documents` holds approved procurement records such as RFP responses, agreements, amendments, and purchase orders.

## Step 6: Verify search indexes

Run the cell below to confirm each restored index has the expected number of chunks:

* `supplier-evidence`: 31 document chunks
* `sourcing-documents`: 16 document chunks

In [ ]:
from azure.search.documents import SearchClient

for index_name in [SUPPLIER_EVIDENCE_INDEX, SOURCING_DOCUMENTS_INDEX]:
    client = SearchClient(
        endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
        index_name=index_name,
        credential=search_credential,
    )
    count = client.get_document_count()
    print(f"{index_name}: {count} document chunks")
    client.close()

## Step 7: Create two indexed knowledge sources

Create one knowledge source for each restored index:

* `supplier-evidence-knowledge-source`: Supplier invoices, shipment evidence, production yield records, and quality records
* `sourcing-documents-knowledge-source`: RFP responses, agreements, amendments, and purchase orders

Both sources search the `chunk` field and return `blob_path`, `title`, `page_number_from`, and `page_number_to` as citation source data. These fields match the portable Caldova index schema.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME = "supplier-evidence-knowledge-source"
SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME = "sourcing-documents-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    FILE_KNOWLEDGE_SOURCE_NAME,
    SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
    SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (
        SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
        SUPPLIER_EVIDENCE_INDEX,
        "Caldova supplier invoices, shipment evidence, production yield records, and quality records",
    ),
    (
        SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
        SOURCING_DOCUMENTS_INDEX,
        "Caldova RFP responses, agreements, amendments, and purchase orders",
    ),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="title"),
                SearchIndexFieldReference(name="page_number_from"),
                SearchIndexFieldReference(name="page_number_to"),
            ],
            search_fields=[SearchIndexFieldReference(name="chunk")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")

## Step 8: Create the multi-source knowledge base

Create a knowledge base that contains the purchasing policy file source and both search index knowledge sources. It uses `output_mode=ANSWER_SYNTHESIS`, so the attached model can answer the original query with grounded citations while keeping each evidence type traceable to its source.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "caldova-supply-chain-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Caldova knowledge base for purchasing policy, supplier evidence, and approved sourcing commitments",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 9: Query the multi-source knowledge base

Ask an operational question that should retrieve excursion and disposition evidence from `supplier-evidence`:

* _"Shipment SHIP-VC-1026-BOS-FRA exceeded its temperature range. What happened, and can the 4,800 units be distributed?"_

The knowledge base also retains the purchasing policy file and approved sourcing documents. After the sample query, try these questions:

* _"Compare Summit Dose's documented CALD-201 commitments across its RFP response, agreement amendment, and purchase order."_
* _"Was lot LOT-MD-1026-A released, and should Caldova pay the $125,600 yield adjustment?"_

Use the activity log and references to verify which source answered each question. Keep policy controls, operational supplier evidence, and approved sourcing commitments distinct in your conclusions.

In [ ]:
from azure.search.documents.knowledgebases.models import (
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential,
)

supplier_evidence_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)
sourcing_documents_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = (
    "Shipment SHIP-VC-1026-BOS-FRA exceeded its temperature range. "
    "What happened, and can the 4,800 units be distributed?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[file_ks_params, supplier_evidence_params, sourcing_documents_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

NameError: name 'KnowledgeBaseRetrievalClient' is not defined

### Review the activity log

For this activity log, you may see a `file` step for purchasing policy and `searchIndex` steps for queries sent to the supplier evidence or sourcing documents indexes.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

The references identify which knowledge source supported each claim. Indexed references have `type: "searchIndex"` and include citation metadata such as `title`, `blob_path`, and page numbers. If purchasing policy is relevant, you may also see a reference with `type: "file"`.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

Look through the references and activity log above. Can you find:

1. Which supplier evidence explains the temperature excursion and its root cause?
2. Which evidence records the quarantine actions and final distribution decision for the 4,800 units?
3. Do indexed references expose `title`, `blob_path`, `page_number_from`, and `page_number_to`?
4. Did the knowledge base keep operational shipment evidence distinct from purchasing policy and approved sourcing commitments?

## Bonus: Query from the Copilot CLI

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or GitHub Copilot CLI.
If you want to query the knowledge base from the Copilot CLI, follow the setup steps in [Copilot CLI sidequest](copilot-cli-sidequest.md) and run the commands below to use the Caldova knowledge base as an MCP server.

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
print(f"copilot mcp add caldova-kb \"{mcp_url}\" --header \"api-key={AZURE_SEARCH_ADMIN_KEY}\"")
print("copilot -i \"What approval thresholds, sourcing routes, and escalation responsibilities does Caldova's purchasing policy define?\"")

## ✅ Mission complete

**What you built:**

* ✓ **File Knowledge Source**: Caldova's purchasing policy uploaded directly for approval, sourcing-route, governance, and escalation questions.
* ✓ **Indexed Knowledge Sources**: Portable supplier evidence and sourcing document indexes using the `chunk` field and citation metadata.
* ✓ **Multi-source Knowledge Base**: One Caldova KB that keeps policy, operational evidence, and approved sourcing commitments traceable to their sources.

➡️ Continue to [Part 2: Add structured data from Fabric IQ](part2-fabric-iq-to-kb.ipynb).